In [0]:
# =============================================================================
# TRAVEL BOOKING SCD2 MERGE PROJECT - DATA QUALITY: CUSTOMER DATA VALIDATION
# =============================================================================
# This notebook performs comprehensive data quality checks on customer data
# Purpose: Validates customer data integrity using PySpark (optimized, no PyDeequ)
# Data Quality: Checks completeness and business rules for customer attributes
# Output: Logs DQ results and raises exceptions for failed validations

from pyspark.sql import functions as F

# =============================================================================
# PARAMETER EXTRACTION WITH DEFAULTS
# =============================================================================

import datetime as _dt
try:
    arrival_date = dbutils.widgets.get("arrival_date")
except Exception:
    arrival_date = _dt.date.today().strftime("%Y-%m-%d")

try:
    catalog = dbutils.widgets.get("catalog")
except Exception:
    catalog = "dbx-external-catalog"

try:
    schema = dbutils.widgets.get("schema")
except Exception:
    schema = "default"

# =============================================================================
# SOURCE DATA PREPARATION
# =============================================================================

src = spark.table(f"`{catalog}`.bronze.customer_inc") \
           .where(F.col("business_date") == F.to_date(F.lit(arrival_date)))

# =============================================================================
# DATA QUALITY CHECKS IMPLEMENTATION (OPTIMIZED - SINGLE PASS)
# =============================================================================

from pyspark.sql import Row

# ---- Single pass aggregation (BEST PRACTICE) ----
agg_df = src.agg(
    F.count("*").alias("row_count"),
    F.sum(F.when(F.col("customer_name").isNull(), 1).otherwise(0)).alias("null_name_count"),
    F.sum(F.when(F.col("customer_address").isNull(), 1).otherwise(0)).alias("null_addr_count"),
    F.sum(F.when(F.col("email").isNull(), 1).otherwise(0)).alias("null_email_count")
)

metrics = agg_df.collect()[0]

dq_results = []

# ---- Check 1: hasSize ----
if metrics["row_count"] > 0:
    dq_results.append(Row(
        check="Customer Data Check",
        check_status="Success",
        constraint="hasSize > 0",
        constraint_status="Success",
        constraint_message=f"Row count = {metrics['row_count']}"
    ))
else:
    dq_results.append(Row(
        check="Customer Data Check",
        check_status="Error",
        constraint="hasSize > 0",
        constraint_status="Failure",
        constraint_message="No data found"
    ))

# ---- Check 2: customer_name completeness ----
if metrics["null_name_count"] == 0:
    dq_results.append(Row(
        check="Customer Data Check",
        check_status="Success",
        constraint="customer_name NOT NULL",
        constraint_status="Success",
        constraint_message="No nulls found"
    ))
else:
    dq_results.append(Row(
        check="Customer Data Check",
        check_status="Error",
        constraint="customer_name NOT NULL",
        constraint_status="Failure",
        constraint_message=f"{metrics['null_name_count']} null values found"
    ))

# ---- Check 3: customer_address completeness ----
if metrics["null_addr_count"] == 0:
    dq_results.append(Row(
        check="Customer Data Check",
        check_status="Success",
        constraint="customer_address NOT NULL",
        constraint_status="Success",
        constraint_message="No nulls found"
    ))
else:
    dq_results.append(Row(
        check="Customer Data Check",
        check_status="Error",
        constraint="customer_address NOT NULL",
        constraint_status="Failure",
        constraint_message=f"{metrics['null_addr_count']} null values found"
    ))

# ---- Check 4: email completeness ----
if metrics["null_email_count"] == 0:
    dq_results.append(Row(
        check="Customer Data Check",
        check_status="Success",
        constraint="email NOT NULL",
        constraint_status="Success",
        constraint_message="No nulls found"
    ))
else:
    dq_results.append(Row(
        check="Customer Data Check",
        check_status="Error",
        constraint="email NOT NULL",
        constraint_status="Failure",
        constraint_message=f"{metrics['null_email_count']} null values found"
    ))

# ---- Convert results to DataFrame ----
df = spark.createDataFrame(
    dq_results,
    ["check", "check_status", "constraint", "constraint_status", "constraint_message"]
)

# ---- Overall result ----
result_status = "Success" if all(r['constraint_status'] == "Success" for r in dq_results) else "Error"

# =============================================================================
# DQ RESULTS STORAGE SETUP
# =============================================================================

spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog}`.ops")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS `{catalog}`.ops.dq_results (
  business_date DATE,
  dataset STRING,
  check_name STRING,
  status STRING,
  constraint STRING,
  message STRING,
  recorded_at TIMESTAMP
) USING DELTA
""")

# =============================================================================
# DQ RESULTS LOGGING
# =============================================================================

out = (df
  .withColumn("business_date", F.to_date(F.lit(arrival_date)))
  .withColumn("dataset", F.lit("customer_inc"))
  .withColumn("recorded_at", F.current_timestamp()))

display(df)

out.select(
    "business_date",
    "dataset",
    F.col("check").alias("check_name"),
    F.col("constraint_status").alias("status"),
    "constraint",
    F.col("constraint_message").alias("message"),
    "recorded_at"
).write.mode("append").option("mergeSchema", "true") \
 .saveAsTable(f"`{catalog}`.ops.dq_results")

# =============================================================================
# DQ VALIDATION AND ERROR HANDLING
# =============================================================================

if result_status != "Success":
    raise ValueError("DQ failed for customers")

print("Customer DQ passed")